# Try GraphPFN on SlaveVoyages

[GraphPFN](https://github.com/yandex-research/graphpfn) is a pretrained model for data shaped as a graph. Here each voyage is a node and voyages are linked when they probably belong together (same ship within 10 years, same first owner, same first captain). It is scored on exactly the same hidden voyages as the other models.

**Runtime > Change runtime type > GPU (T4).** GraphPFN needs an older Python (3.11) and PyTorch 2.4, but Colab now ships Python 3.13. So this notebook makes a separate Python 3.11 environment with `uv` and runs everything inside it. You need the data file `tastdb-exp-2019.csv` (see DATA.md in the repo).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Ememobong28/slavevoyages-project.git"
!git clone -q {REPO_URL} repo
%cd repo

## 2. Make a Python 3.11 environment and install GraphPFN

Takes a few minutes. If a line fails with a network (SSL) error, run the cell again; the DGL download site sometimes drops the connection.

In [ ]:
!pip install -q uv
!uv venv --python 3.11 /content/gp311
PY = "/content/gp311/bin/python"
!uv pip install --python {PY} -q -r requirements.txt
!uv pip install --python {PY} -q "torch==2.4.1" --index-url https://download.pytorch.org/whl/cu118
!uv pip install --python {PY} -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu118/repo.html
!uv pip install --python {PY} -q graphpfn

In [ ]:
!{PY} -c "import sys, torch, dgl, graphpfn; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'dgl', dgl.__version__)"

If the line above prints a Python 3.11 version, a torch 2.4 version and `cuda True`, you are ready. If not, copy the output and the red error lines.

## 3. Add the data and build the database

In [ ]:
from google.colab import files
import os
if not os.path.exists('tastdb-exp-2019.csv'):
    files.upload()                                   # choose tastdb-exp-2019.csv
!{PY} voyage_db.py sqlite tastdb-exp-2019.csv voyages.sqlite
!{PY} tests/test_framework.py

## 4. One quick run (tonnage, random split)

`none` is GraphPFN with no links. The other two add links. Comparing them shows what the links add.

In [ ]:
!{PY} graphpfn_cli.py --targets TONNAGE --schemes random --seeds 1 --graphs none,ship10,ship10_owner_captain

## 5. All splits, tonnage and nationality

Same five random splits (seeds 42-46) and the time-based split as the other models. This takes a long time on a free GPU; start with a smaller run if you are short of time (for example `--seeds 2`).

In [ ]:
!{PY} graphpfn_cli.py --targets TONNAGE,NATIONAL_GROUP --schemes random,temporal --seeds 5

## 6. Optional: fine-tuning

Updates the model on this data. Slower; it is the setting the GraphPFN authors recommend.

In [ ]:
!{PY} graphpfn_cli.py --targets TONNAGE --schemes random --seeds 1 --graphs ship10_owner_captain --finetune

## 7. Download the results

Send the zip back or commit its contents to `results/graphpfn/`.

In [ ]:
!zip -qr graphpfn_results.zip results/graphpfn
files.download('graphpfn_results.zip')